# NLI Checks

Validates Phase 7 (NLI integration) before moving to Phase 8.

Goals:
- load the NLI model
- run 20 hand-picked sanity pairs (support, refute, unrelated, ambiguous)
- inspect label + confidence + full score distribution
- verify relevance filtering drops low-overlap snippets
- run StanceService in both single-model and cascade mode
- confirm cascade overrides false refutes from the fast model
- confirm cache works (second run is faster)

In [1]:
import sys
sys.path.insert(0, "..")

from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

from backend.app.models.nli_model import NLIModel, NLIResult
from backend.app.services.stance_service import StanceService
from backend.app.services.cache_service import CacheService
from backend.app.preprocessing.snippet_extractor import score_sentence_relevance
from backend.app.utils.constants import (
    NLI_MODEL_NAME,
    NLI_CONFIRM_MODEL_NAME,
    NLI_DEVICE,
    NLI_CONFIDENCE_THRESHOLD,
    SNIPPET_MIN_RELEVANCE_SCORE,
    DEFAULT_RETRIEVAL_CACHE_DIR,
)

CACHE_DIR = PROJECT_ROOT / DEFAULT_RETRIEVAL_CACHE_DIR

print(f"Fast model      : {NLI_MODEL_NAME}")
print(f"Confirm model   : {NLI_CONFIRM_MODEL_NAME}")
print(f"Device          : {NLI_DEVICE}")
print(f"NLI threshold   : {NLI_CONFIDENCE_THRESHOLD}")
print(f"Min relevance   : {SNIPPET_MIN_RELEVANCE_SCORE}")
print(f"Cache dir       : {CACHE_DIR}")

/Users/vraj21/Desktop/Projects/Fact Checker/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fast model      : cross-encoder/nli-deberta-v3-small
Confirm model   : MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Device          : cpu
NLI threshold   : 0.5
Min relevance   : 0.15
Cache dir       : /Users/vraj21/Desktop/Projects/Fact Checker/data/cache/retrieval_results


## Load Models

In [2]:
print("Loading fast model...")
nli_fast = NLIModel(model_name=NLI_MODEL_NAME, device=NLI_DEVICE)
print(f"Fast model labels   : {nli_fast._id2label}")

print("\nLoading confirm model...")
nli_confirm = NLIModel(model_name=NLI_CONFIRM_MODEL_NAME, device=NLI_DEVICE)
print(f"Confirm model labels: {nli_confirm._id2label}")

print("\nBoth models loaded.")

Loading fast model...


Loading weights: 100%|██████████| 106/106 [00:00<00:00, 2013.94it/s, Materializing param=pooler.dense.weight]                                     
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fast model labels   : {0: 'contradiction', 1: 'entailment', 2: 'neutral'}

Loading confirm model...


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 2176.56it/s, Materializing param=pooler.dense.weight]                                       
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Confirm model labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}

Both models loaded.


## Relevance Filter Check

Verify that `SNIPPET_MIN_RELEVANCE_SCORE = 0.15` correctly filters single-term matches
while keeping multi-term matches.

In [3]:
CLAIM = "The Eiffel Tower is located in Paris, France."

relevance_cases = [
    # (snippet, expected_outcome)
    ("France has paid tribute to victims of the 2015 Paris attacks.",        "single-term: should be filtered"),
    ("Crystal Palace drew 0-0 with Larnaca in the Conference League.",        "zero-overlap: should be filtered"),
    ("Two women in Texas were arrested after disguising drones as birds.",    "zero-overlap: should be filtered"),
    ("The Eiffel Tower in Paris attracts millions of visitors each year.",    "multi-term: should pass"),
    ("The Eiffel Tower is a wrought-iron lattice tower in Paris, France.",   "multi-term: should pass"),
    ("The Stade de France is located just north of Paris.",                   "multi-term: should pass"),
]

print(f"{'Score':<8} {'Pass?':<7} {'Expected'}  →  snippet[:60]")
print("-" * 90)
for snippet, expected in relevance_cases:
    score = score_sentence_relevance(CLAIM, snippet)
    passes = score >= SNIPPET_MIN_RELEVANCE_SCORE
    flag = "PASS" if passes else "DROP"
    print(f"{score:<8.3f} {flag:<7} {expected[:40]}  →  {snippet[:60]}")

print(f"\nThreshold: {SNIPPET_MIN_RELEVANCE_SCORE}")
print("Single-term matches score ~0.06 (below threshold).")
print("Multi-term matches score >= 0.15 (above threshold).")

Score    Pass?   Expected  →  snippet[:60]
------------------------------------------------------------------------------------------
0.400    PASS    single-term: should be filtered  →  France has paid tribute to victims of the 2015 Paris attacks
0.000    DROP    zero-overlap: should be filtered  →  Crystal Palace drew 0-0 with Larnaca in the Conference Leagu
0.000    DROP    zero-overlap: should be filtered  →  Two women in Texas were arrested after disguising drones as 
0.600    PASS    multi-term: should pass  →  The Eiffel Tower in Paris attracts millions of visitors each
0.800    PASS    multi-term: should pass  →  The Eiffel Tower is a wrought-iron lattice tower in Paris, F
0.600    PASS    multi-term: should pass  →  The Stade de France is located just north of Paris.

Threshold: 0.15
Single-term matches score ~0.06 (below threshold).
Multi-term matches score >= 0.15 (above threshold).


## 20 Sanity Pairs

Four categories: support, refute, unrelated, ambiguous

In [4]:
SANITY_PAIRS = [
    # support
    ("The Eiffel Tower is located in Paris, France.", "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.", "support"),
    ("Water boils at 100 degrees Celsius at sea level.", "At standard atmospheric pressure, water boils at 100 degrees Celsius.", "support"),
    ("Shakespeare wrote Romeo and Juliet.", "Romeo and Juliet is a tragedy written by William Shakespeare.", "support"),
    ("The human body has 206 bones.", "By adulthood the human skeleton is made up of 206 bones.", "support"),
    ("Neil Armstrong was the first person to walk on the Moon.", "On July 20, 1969, Neil Armstrong became the first human to set foot on the Moon.", "support"),
    # refute
    ("The Earth is flat.", "The Earth is an oblate spheroid, slightly flattened at the poles.", "refute"),
    ("Adolf Hitler was born in Germany.", "Adolf Hitler was born in Braunau am Inn, Austria, on April 20, 1889.", "refute"),
    ("The currency of Japan is the Yuan.", "The currency of Japan is the yen, not the yuan which is used in China.", "refute"),
    ("The Amazon is the longest river in the world.", "The Nile River is generally considered the longest river in the world, not the Amazon.", "refute"),
    ("The Great Wall of China is visible from space with the naked eye.", "Astronauts have confirmed the Great Wall cannot be seen from space with the naked eye.", "refute"),
    # unrelated
    ("The Eiffel Tower is located in Paris, France.", "Germany is expecting heavy snowfalls of up to 20cm after record winds.", "unrelated"),
    ("The Eiffel Tower is located in Paris, France.", "Two women in Texas were arrested after disguising drones to look like birds.", "unrelated"),
    ("Shakespeare wrote Romeo and Juliet.", "The stock market fell sharply on Friday amid concerns about inflation.", "unrelated"),
    ("Water boils at 100 degrees Celsius.", "A new fashion collection debuted at Paris Fashion Week.", "unrelated"),
    ("The Amazon is the longest river in the world.", "Apple announced its latest iPhone model at a product launch event.", "unrelated"),
    # ambiguous
    ("Henri Christophe built a palace in Milot.", "Henri Christophe was a key leader of the Haitian Revolution.", "ambiguous"),
    ("The Great Wall of China is visible from space.", "The Great Wall of China stretches over 13,000 miles across northern China.", "ambiguous"),
    ("Neil Armstrong landed on the Moon in 1969.", "NASA's Apollo program achieved several milestones in space exploration during the 1960s.", "ambiguous"),
    ("The Eiffel Tower is the tallest structure in Paris.", "The Eiffel Tower stands 330 metres tall including its broadcast antenna.", "ambiguous"),
    ("The Amazon river flows through Brazil.", "The Amazon basin covers about 40% of South America.", "ambiguous"),
]

### Fast model predictions

In [5]:
claims   = [p[0] for p in SANITY_PAIRS]
snippets = [p[1] for p in SANITY_PAIRS]
expected = [p[2] for p in SANITY_PAIRS]

print("Running fast model on 20 pairs...\n")
fast_results = []
for claim, snippet, _ in SANITY_PAIRS:
    r = nli_fast.predict(claim, [snippet])
    fast_results.append(r[0])

print(f"{'#':<3} {'Expected':<10} {'Fast':<18} {'Conf':<7} {'Supp':<7} {'Refu':<7} {'NEI':<7}")
print("-" * 65)
for i, (r, exp) in enumerate(zip(fast_results, expected), start=1):
    s  = r.scores.get('supports', 0)
    rf = r.scores.get('refutes', 0)
    n  = r.scores.get('not_enough_info', 0)
    print(f"{i:<3} {exp:<10} {r.label:<18} {r.confidence:<7.3f} {s:<7.3f} {rf:<7.3f} {n:<7.3f}")

Running fast model on 20 pairs...

#   Expected   Fast               Conf    Supp    Refu    NEI    
-----------------------------------------------------------------
1   support    supports           0.988   0.988   0.001   0.012  
2   support    refutes            0.802   0.030   0.802   0.168  
3   support    supports           0.998   0.998   0.000   0.002  
4   support    supports           0.996   0.996   0.000   0.003  
5   support    supports           0.998   0.998   0.000   0.002  
6   refute     supports           0.965   0.965   0.008   0.027  
7   refute     refutes            1.000   0.000   1.000   0.000  
8   refute     refutes            1.000   0.000   1.000   0.000  
9   refute     refutes            0.998   0.001   0.998   0.001  
10  refute     refutes            0.993   0.001   0.993   0.006  
11  unrelated  refutes            0.999   0.000   0.999   0.001  
12  unrelated  refutes            0.999   0.000   0.999   0.001  
13  unrelated  refutes            0.997  

### Cascade: confirm model on fast model's supports/refutes

Only pairs where the fast model predicted `supports` or `refutes` are sent to the confirm model.
Ambiguous/unrelated pairs (predicted `not_enough_info`) skip the confirm model entirely.

In [6]:
to_confirm_indices = [i for i, r in enumerate(fast_results) if r.label in ("supports", "refutes")]
print(f"Fast model predictions to confirm: {len(to_confirm_indices)} / {len(SANITY_PAIRS)}")
print(f"Skipped (not_enough_info): {len(SANITY_PAIRS) - len(to_confirm_indices)}\n")

confirm_results = list(fast_results)  # start as copy

if to_confirm_indices:
    confirm_snippets = [snippets[i] for i in to_confirm_indices]
    confirm_claims   = [claims[i]   for i in to_confirm_indices]
    confirm_preds    = [nli_confirm.predict(c, [s])[0] for c, s in zip(confirm_claims, confirm_snippets)]

    for idx, pred in zip(to_confirm_indices, confirm_preds):
        confirm_results[idx] = pred

print(f"{'#':<3} {'Expected':<10} {'Fast':<18} {'Confirm':<18} {'Override?'}")
print("-" * 72)
overrides = 0
for i, (fast, confirm, exp) in enumerate(zip(fast_results, confirm_results, expected), start=1):
    changed = fast.label != confirm.label
    if changed:
        overrides += 1
    flag = "<-- OVERRIDE" if changed else ""
    print(f"{i:<3} {exp:<10} {fast.label:<18} {confirm.label:<18} {flag}")

print(f"\nTotal overrides: {overrides}")

Fast model predictions to confirm: 16 / 20
Skipped (not_enough_info): 4

#   Expected   Fast               Confirm            Override?
------------------------------------------------------------------------
1   support    supports           supports           
2   support    refutes            not_enough_info    <-- OVERRIDE
3   support    supports           supports           
4   support    supports           supports           
5   support    supports           supports           
6   refute     supports           refutes            <-- OVERRIDE
7   refute     refutes            refutes            
8   refute     refutes            refutes            
9   refute     refutes            not_enough_info    <-- OVERRIDE
10  refute     refutes            refutes            
11  unrelated  refutes            not_enough_info    <-- OVERRIDE
12  unrelated  refutes            not_enough_info    <-- OVERRIDE
13  unrelated  refutes            not_enough_info    <-- OVERRIDE
14  unrelated  re

## Distribution Check

In [7]:
for model_name, results in [("Fast", fast_results), ("Confirm", confirm_results)]:
    counts = {"supports": 0, "refutes": 0, "not_enough_info": 0}
    for r in results:
        counts[r.label] = counts.get(r.label, 0) + 1
    print(f"{model_name} model distribution: {counts}")

assert confirm_results[0].label == "supports", "Pair 1 (Eiffel Tower) should be supports"
assert confirm_results[5].label == "refutes",  "Pair 6 (flat Earth) should be refutes"
print("\nKey assertion checks passed.")

Fast model distribution: {'supports': 5, 'refutes': 11, 'not_enough_info': 4}
Confirm model distribution: {'supports': 4, 'refutes': 4, 'not_enough_info': 12}

Key assertion checks passed.


## Pipeline Integration Check

Run the full pipeline (retrieval → expansion → stance) on a real claim
with cascade mode enabled.

In [8]:
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

from backend.app.retrieval.wikipedia_retriever import WikipediaRetriever
from backend.app.retrieval.factcheck_retriever import FactCheckRetriever
from backend.app.retrieval.guardian_retriever import GuardianRetriever
from backend.app.retrieval.newsapi_retriever import NewsApiRetriever
from backend.app.retrieval.gdelt_retriever import GDELTRetriever
from backend.app.retrieval.retriever_registry import RetrieverRegistry
from backend.app.services.ranking_service import RankingService
from backend.app.services.retrieval_service import RetrievalService
from backend.app.services.evidence_expansion_service import EvidenceExpansionService
from backend.app.utils.constants import (
    DEFAULT_PROCESSED_DIR,
    FEVER_EVIDENCE_SNIPPETS_OUTPUT_FILENAME,
)

registry = RetrieverRegistry()
registry.register(WikipediaRetriever(
    evidence_snippets_path=PROJECT_ROOT / DEFAULT_PROCESSED_DIR / FEVER_EVIDENCE_SNIPPETS_OUTPUT_FILENAME
))
registry.register(FactCheckRetriever(api_key=os.getenv("FACTCHECK_API_KEY")))
registry.register(GuardianRetriever(api_key=os.getenv("GUARDIAN_API_KEY")))
registry.register(NewsApiRetriever(api_key=os.getenv("NEWSAPI_KEY")))
registry.register(GDELTRetriever())

cache = CacheService(CACHE_DIR)
retrieval_svc = RetrievalService(
    registry=registry, cache=cache,
    ranking=RankingService(), expansion=EvidenceExpansionService(),
)

# Cascade mode: fast model pre-filters, confirm model verifies supports/refutes
stance_svc = StanceService(
    model=nli_fast,
    cache=cache,
    confirm_model=nli_confirm,
)
print("Services ready (cascade mode enabled).")
print(f"Cache dir: {CACHE_DIR}")

Services ready (cascade mode enabled).
Cache dir: /Users/vraj21/Desktop/Projects/Fact Checker/data/cache/retrieval_results


In [9]:
TEST_CLAIM = "The Eiffel Tower is located in Paris, France."

sources = retrieval_svc.retrieve(TEST_CLAIM, max_results=10, use_cache=False)
print(f"Retrieved {len(sources)} sources (after relevance filter)")

classified = stance_svc.classify(TEST_CLAIM, sources)

print(f"\n{'#':<3} {'Type':<12} {'Stance':<14} {'Title'}")
print("-" * 80)
for i, s in enumerate(classified, start=1):
    print(f"{i:<3} {s.source_type:<12} {str(s.stance_hint):<14} {s.title[:50]}")

stance_counts = {}
for s in classified:
    k = s.stance_hint or "none"
    stance_counts[k] = stance_counts.get(k, 0) + 1
print(f"\nSummary: {stance_counts}")

Retriever 'newsapi' failed for query='The Eiffel Tower is located in Paris, France.': HTTPSConnectionPool(host='newsapi.org', port=443): Max retries exceeded with url: /v2/everything?q=The+Eiffel+Tower+is+located+in+Paris%2C+France.&language=en&sortBy=relevancy&pageSize=25&page=1&searchIn=title%2Cdescription (Caused by NameResolutionError("HTTPSConnection(host='newsapi.org', port=443): Failed to resolve 'newsapi.org' ([Errno 8] nodename nor servname provided, or not known)"))
Partial retrieval: 1 retrievers failed (['newsapi']). Proceeding with 16 raw results from 4 sources.


Retrieved 3 sources (after relevance filter)

#   Type         Stance         Title
--------------------------------------------------------------------------------
1   guardian     insufficient   ‘The pain remains’: France remembers victims of 20
2   wikipedia    insufficient   Wikipedia: Paris (sentence 25)
3   wikipedia    insufficient   Wikipedia: 4th_arrondissement_of_Paris (sentence 0

Summary: {'insufficient': 3}


## Cache Check

Cascade results are cached with a key that includes `confirm_model`.
Running classify a second time should be near-instant.

In [10]:
import time

t0 = time.time()
classified_cached = stance_svc.classify(TEST_CLAIM, sources)
t1 = time.time()

print(f"Second classify() call: {(t1 - t0)*1000:.1f} ms (should be near-zero — all from cache)")

for a, b in zip(classified, classified_cached):
    assert a.stance_hint == b.stance_hint, f"Cache mismatch on source_id={a.source_id}"

print("Cache check passed — stance_hints match between first and second run.")

Second classify() call: 1.3 ms (should be near-zero — all from cache)
Cache check passed — stance_hints match between first and second run.


## Final Checklist

Before moving to Phase 8, confirm:

- Both models load without errors
- `predict()` returns one `NLIResult` per snippet
- Labels are one of: `supports`, `refutes`, `not_enough_info`
- `scores` values sum to approximately 1.0
- Relevance filter drops single-term matches (score < 0.15) before NLI
- Cascade: confirm model only runs on supports/refutes from fast model
- Cascade: overrides false refutes (e.g. unrelated sources) to `not_enough_info`
- `StanceService.classify()` returns same-length list with `stance_hint` populated
- Second `classify()` call is significantly faster (NLI cache hit)
- Sources without snippets pass through with `stance_hint=None`